# FaithfulMed Extractor Agent

This notebook is the working prototype for the Extractor agent. It loads MTSamples notes, splits the transcription into clinical sections, and extracts structured facts that later agents can use.

The goal is to convert raw transcribed medical text into a clean list of atomic facts such as diagnoses, medications, allergies, procedures, symptoms, and lab values.

In [ ]:
from pathlib import Path
import csv
import json
import re

DATA_PATH = Path.cwd().parent / "data" / "mtsamples - mtsamples.csv"
print(DATA_PATH.exists())
print(DATA_PATH)

In [ ]:
def clean_text(text: str) -> str:
    value = re.sub(r"\s+", " ", text or "").strip()
    value = re.sub(r"^[\s,;:.-]+|[\s,;:.-]+$", "", value)
    return value


def normalize_fact_value(text: str) -> str:
    value = clean_text(text)
    value = re.sub(r"\s+([,.;:])", r"\1", value)
    value = re.sub(r"([,.;:]){2,}", ".", value)
    value = re.sub(r"\s+\.", ".", value)
    value = re.sub(r"\.+$", "", value)
    value = re.sub(r"^\.+", "", value)
    return clean_text(value)


def get_sections(transcription: str) -> dict:
    if not transcription:
        return {}

    pattern = re.compile(
        r"(?i)\b([A-Z][A-Z /-]{2,}?)\s*:\s*|\b([A-Z][A-Z /-]{2,}?)\s*[,\n]"
    )
    matches = list(pattern.finditer(transcription))
    sections = {}

    for idx, match in enumerate(matches):
        label = (match.group(1) or match.group(2) or "").strip()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(transcription)
        value = clean_text(transcription[start:end])
        if label:
            sections[label.upper()] = value

    if not sections:
        sections["FULL_TEXT"] = clean_text(transcription)

    return sections


def extract_clinical_atoms(transcription: str) -> list[dict]:
    sections = get_sections(transcription)
    atoms = []

    def add_atom(atom_type: str, value: str, source: str = "") -> None:
        if not value:
            return
        normalized_value = normalize_fact_value(value)
        if not normalized_value:
            return
        atoms.append({
            "type": atom_type,
            "value": normalized_value,
            "source": clean_text(source),
        })

    for key in ["ASSESSMENT", "IMPRESSION", "DIAGNOSIS", "POSTOPERATIVE DIAGNOSIS", "PREOPERATIVE DIAGNOSIS"]:
        if key in sections:
            add_atom("diagnosis", sections[key], key)

    for key in ["MEDICATIONS", "CURRENT MEDICATIONS", "HOME MEDICATIONS"]:
        if key in sections:
            add_atom("medication", sections[key], key)

    for key in ["ALLERGIES", "ALLERGIES TO MEDICATIONS"]:
        if key in sections:
            add_atom("allergy", sections[key], key)

    for key in ["PROCEDURE", "PROCEDURE PERFORMED", "OPERATION", "OPERATION PERFORMED", "PROCEDURES"]:
        if key in sections:
            add_atom("procedure", sections[key], key)

    for key in ["CHIEF COMPLAINT", "HISTORY OF PRESENT ILLNESS", "REVIEW OF SYSTEMS", "SUBJECTIVE"]:
        if key in sections:
            text = sections[key]
            if len(text) < 300:
                add_atom("symptom_or_history", text, key)

    lab_pattern = re.compile(
        r"([A-Z][A-Za-z ]+\s*[:=]\s*[0-9]+\.?[0-9]*\s*(?:mg/dL|mmHg|%|mEq/L|mmol/L|units|cm|kg|lbs|bpm|C|F|mm)?)"
    )
    for field_name, value in sections.items():
        for match in lab_pattern.finditer(value):
            add_atom("lab_value", match.group(1), field_name)

    if not atoms:
        add_atom("raw_text", transcription, "FULL_TEXT")

    return atoms


def load_mtsamples(path: Path = DATA_PATH) -> list[dict]:
    with open(path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        return list(reader)

In [ ]:
rows = load_mtsamples()
sample = rows[0]
transcription = sample.get("transcription", "")
print(f"Loaded {len(rows)} MTSamples records.")
print("Sample specialty:", sample.get("medical_specialty", "Unknown"))
print("Sample name:", sample.get("sample_name", "Unknown"))
print("\nExtracted atoms from the first note:\n")
print(json.dumps(extract_clinical_atoms(transcription)[:8], indent=2, ensure_ascii=False))